# Stage 1: Data Collection and Data Cleaning

### Project Structure

```bash
CS5246Project/
    │
    ├─ data/
    │   ├─ inverted_index/
    │   │   ├─ inverted_index_tfidf_titles.json
    │   │   └─ inverted_index_tfidf_fulltext.json
    │   │
    │   ├─ models/
    │   │   ├─ sentence_bert_model/
    │   │   ├─ bm25_comments_model.joblib
    │   │   ├─ bm25_fulltext_model.joblib
    │   │   ├─ bm25_titles_model.joblib
    │   │   ├─ tfidf_comments_vectorizer.joblib
    │   │   └─ tfidf_posts_vectorizer.joblib
    │   │
    │   ├─ vector_database/
    │   │   ├─ bert_comments_embeddings.npy
    │   │   ├─ bert_contents_embeddings.npy
    │   │   ├─ bert_fulltext_embeddings.npy
    │   │   ├─ bert_titles_embeddings.npy
    │   │   ├─ bm25_comments.npz
    │   │   ├─ bm25_contents.npz
    │   │   ├─ bm25_fulltext.npz
    │   │   ├─ bm25_titles.npz
    │   │   ├─ tfidf_comments.npz
    │   │   ├─ tfidf_contents.npz
    │   │   ├─ tfidf_fulltext.npz
    │   │   └─ tfidf_titles.npz
    │   │
    │   ├─ PostVault.csv
    │   ├─ CommentVault.csv
    │   └─ raw_data/
    │
    ├─ sentiment_plots/
    │   ├─ emotion_plots/
    │   ├─ emotion_dashboard.py
    │   └─ plot_emotion_summary.py
    │
    ├─ Stage_0_Introduction.ipynb                               
-----------------------------------------------------------------            
│   ├─ Stage_1_Data_Collection_and_Data_Cleaning.ipynb          │
-----------------------------------------------------------------
    ├─ Stage_2_POS_and_NER_Tagging.ipynb
    ├─ Stage_3_Singlish_Normalisation.ipynb
    ├─ Stage_4_Singlish_to_English_Conversion.ipynb     
    ├─ Stage_5_Common_Normalisation.ipynb
    ├─ Stage_6_Vector_Space_Model_and_Inverted_Index.ipynb
    ├─ Stage_7_Sentiment_Analysis.ipynb
    ├─ Stage_8_Clustering_and_Visualization.ipynb       
    ├─ Stage_9_Document_Search.ipynb
    └─ utilities/
        │
        ├─ pp_class.py
        ├─ singlish_dictionary.json
        ├─ singlish_regex_to_text.txt
        └─ slang_dictionary.csv
```

## To access the mounted folder directly.

### Step 1: Add the Shared Folder to your Google Drive (if you haven't already)

1.  **Open Google Drive** (drive.google.com) in your web browser.
2.  Go to the **'Shared with me'** section.
3.  Locate the folder that was shared with you.
4.  **Right-click** on the shared folder.
5.  Select **'Add shortcut to Drive'** (or 'Add to My Drive', depending on the interface). Choose a location within 'My Drive' if prompted, or simply add it to the root of 'My Drive'.

This creates a shortcut to the shared folder in your 'My Drive', making it accessible when Colab mounts your personal Drive.
### Step 2: Access the Shared Folder in Colab (No action needed)

Once your Drive is mounted, you can navigate to the shared folder. If you added a shortcut, it will appear in your 'My Drive' just like any other folder. The path will typically be `/content/gdrive/MyDrive/YourSharedFolderName`.

It should be able to run the script below already

In [ ]:
# -----------
# Mount Drive
# -----------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Dependencies

### Install Dependencies

In [ ]:
!pip install emoji ftfy langdetect import-ipynb contractions

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.7 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=6850cacef2de9a858030783dabcf6bd6017206600c9b60dfe644bf8255f9e1fb
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


### Change Directory

In [ ]:
import os

# Path
# ----

FOLDER_PATH: str = "/content/drive/MyDrive/CS5246Project/"

os.chdir(FOLDER_PATH)

### Import Essential Library

In [ ]:
import import_ipynb
import pandas as pd
from typing import Any
from Stage_0_Introduction import display_first_text_column_head, save_dfs_to_csv, output_base_dir, save_single_df_to_csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Reddit Preprocessor Class

In [ ]:
from __future__ import annotations

import glob
import html
import logging
import re
from typing import Any

import contractions
import emoji
import ftfy
import pandas as pd
from langdetect import LangDetectException, detect

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)


class RedditPreprocessor:
    """
    Single-class Reddit preprocessing pipeline for posts and comments.
    """

    REMOVED_MARKERS = {"[removed]", "[deleted]", "[deleted by user]"}
    KNOWN_BOTS = {
        "AutoModerator",
        "SG_wormsbot",
        "RepostSleuthBot",
        "RemindMeBot",
        "reddit_repost_sleuth",
        "TotesMessenger",
        "converter-bot",
    }
    SCORE_BINS = [-float("inf"), 0, 5, 25, 100, float("inf")]
    SCORE_LABELS = ["negative", "low", "medium", "high", "viral"]

    POSTS_CANONICAL_COLS = [
        "id",
        "title",
        "selftext",
        "author",
        "score",
        "upvote_ratio",
        "num_comments",
        "created_utc",
        "edited",
        "distinguished",
        "stickied",
        "over_18",
        "spoiler",
        "locked",
        "archived",
        "is_original_content",
        "is_self",
        "permalink",
        "url",
        "domain",
        "subreddit",
        "subreddit_id",
        "link_flair_text",
        "link_flair_css_class",
        "author_flair_text",
        "author_flair_css_class",
        "gilded",
        "total_awards_received",
        "is_video",
        "media",
        "thumbnail",
        "post_hint",
        "treatment_tags",
        "secure_media",
        "preview",
        "downloaded_media",
    ]

    COMMENTS_CANONICAL_COLS = [
        "id",
        "parent_id",
        "post_id",
        "body",
        "author",
        "score",
        "created_utc",
        "edited",
        "distinguished",
        "stickied",
        "is_submitter",
        "author_flair_text",
        "author_flair_css_class",
        "gilded",
        "total_awards_received",
        "permalink",
        "subreddit",
        "subreddit_id",
        "depth",
        "controversiality",
        "collapsed",
        "collapsed_reason",
        "locked",
    ]

    _RE_URL = re.compile(r"https?://\S+|www\.\S+")
    _RE_MENTIONS = re.compile(r"\b[ur]/\w+")
    _RE_MD_LINK = re.compile(r"\[([^\]]+)\]\([^)]+\)")
    _RE_MD_IMAGE = re.compile(r"!\[[^\]]*\]\([^)]+\)")
    _RE_BOLD_ITALIC = re.compile(r"\*{1,3}(.+?)\*{1,3}")
    _RE_STRIKETHROUGH = re.compile(r"~~(.+?)~~")
    _RE_INLINE_CODE = re.compile(r"`[^`]+`")
    _RE_CODE_BLOCK = re.compile(r"```[\s\S]*?```")
    _RE_BLOCKQUOTE = re.compile(r"^>+\s?", re.MULTILINE)
    _RE_HEADER = re.compile(r"^#{1,6}\s+", re.MULTILINE)
    _RE_HR = re.compile(r"^[-*_]{3,}\s*$", re.MULTILINE)
    _RE_MULTI_UNDERSCORE = re.compile(r"_+")
    _RE_ELONGATION       = re.compile(r"(.)\1{2,}")
    _RE_REPEATED_PUNCT   = re.compile(r"([!?.])\1+")
    _RE_NON_ALPHA        = re.compile(r"[^a-z0-9\s]")

    # ASCII emoticons — matched between non-word boundaries or string edges.
    # Order matters: longer/more specific patterns first.
    _ASCII_EMOJI_MAP: list[tuple[re.Pattern, str]] = [
        # crying / very sad
        (re.compile(r"(?<!\w)[;:]['`]-?\((?!\w)"),        "crying"),
        (re.compile(r"(?<!\w)t[_.]t(?<!\w)"),              "crying"),
        (re.compile(r"(?<!\w)q[_.]q(?<!\w)"),              "crying"),
        # angry
        (re.compile(r"(?<!\w)>:-\((?!\w)"),              "angry"),
        (re.compile(r"(?<!\w)>_<(?<!\w)"),                 "frustrated"),
        # surprised / shocked
        (re.compile(r"(?<!\w):-?[oO](?!\w)"),             "surprised"),
        (re.compile(r"(?<!\w)[oO][_.-][oO](?!\w)"),       "shocked"),
        (re.compile(r"(?<!\w)[oO][_.-][oO](?!\w)"),       "shocked"),
        # grinning / very happy
        (re.compile(r"(?<!\w)x[Dd](?!\w)"),               "grinning"),
        (re.compile(r"(?<!\w):-?[dD](?!\w)"),             "grinning"),
        (re.compile(r"(?<!\w)=\s?[dD](?!\w)"),            "grinning"),
        (re.compile(r"(?<!\w)\^[_.-]?\^(?!\w)"),         "happy"),
        # winking
        (re.compile(r"(?<!\w);-?\)(?!\w)"),               "winking"),
        (re.compile(r"(?<!\w);-?[dD](?!\w)"),             "winking grin"),
        # happy / smile
        (re.compile(r"(?<!\w)[=:]-?\)(?!\w)"),            "happy"),
        (re.compile(r"(?<!\w>[=:])\)(?!\w)"),             "happy"),
        # sad / frown
        (re.compile(r"(?<!\w)[=:]-?\((?!\w)"),            "sad"),
        (re.compile(r"(?<!\w>[=:])\((?!\w)"),             "sad"),
        # skeptical / unsure
        (re.compile(r"(?<!\w)[=:]-?[/\\](?<!\w)"),         "skeptical"),
        # tongue / playful
        (re.compile(r"(?<!\w)[=:]-?[pP](?!\w)"),         "playful"),
        # neutral
        (re.compile(r"(?<!\w)[=:]-?\|(?<!\w)"),            "neutral"),
        # love / heartbreak
        (re.compile(r"(?<!\w)<3(?<!\w)"),                  "love"),
        (re.compile(r"(?<!\w)</3(?<!\w)"),                 "heartbreak"),
    ]

    def __init__(self, min_words: int = 3, lang_detect: bool = True) -> None:
        self.min_words = min_words
        self.lang_detect = lang_detect

    def harmonise_schema(self, df: pd.DataFrame, canonical_cols: list[str]) -> pd.DataFrame:
        out = df.copy()
        for col in canonical_cols:
            if col not in out.columns:
                out[col] = None
        return out[canonical_cols]

    def dedup(self, df: pd.DataFrame, id_col: str = "id") -> pd.DataFrame:
        before = len(df)
        out = df.drop_duplicates(subset=[id_col])
        log.info("Dedup: %d -> %d rows (removed %d)", before, len(out), before - len(out))
        return out

    def drop_removed(self, df: pd.DataFrame, text_col: str) -> pd.DataFrame:
        before = len(df)
        mask = df[text_col].astype(str).str.strip().isin(self.REMOVED_MARKERS)
        out = df[~mask]
        log.info("Drop removed: %d -> %d rows (removed %d)", before, len(out), before - len(out))
        return out

    def drop_bots_and_mods(self, df: pd.DataFrame) -> pd.DataFrame:
        before = len(df)
        is_bot = df["author"].isin(self.KNOWN_BOTS)
        is_mod = df["distinguished"].astype(str).str.lower().isin({"moderator", "admin"})
        is_stickied = df.get("stickied", pd.Series(False, index=df.index)).astype(bool)
        out = df[~(is_bot | is_mod | is_stickied)]
        log.info("Drop bots/mods/stickied: %d -> %d rows (removed %d)", before, len(out), before - len(out))
        return out

    def drop_nsfw(self, df: pd.DataFrame) -> pd.DataFrame:
        before = len(df)
        is_nsfw = df["over_18"].astype(str).str.lower() == "true"
        out = df[~is_nsfw]
        log.info("Drop NSFW: %d -> %d rows (removed %d)", before, len(out), before - len(out))
        return out

    def strip_markdown(self, text: str) -> str:
        text = html.unescape(text)
        text = self._RE_CODE_BLOCK.sub(" ", text)
        text = self._RE_MD_IMAGE.sub(" ", text)
        text = self._RE_MD_LINK.sub(r"\1", text)
        text = self._RE_URL.sub(" ", text)
        text = self._RE_MENTIONS.sub(" ", text)
        text = self._RE_BOLD_ITALIC.sub(r"\1", text)
        text = self._RE_STRIKETHROUGH.sub(r"\1", text)
        text = self._RE_INLINE_CODE.sub(" ", text)
        text = self._RE_BLOCKQUOTE.sub(" ", text)
        text = self._RE_HEADER.sub(" ", text)
        text = self._RE_HR.sub(" ", text)
        return text

    @staticmethod
    def fix_encoding(text: str) -> str:
        return ftfy.fix_text(text)

    @staticmethod
    def normalise_text(text: str) -> str:
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()

    def resolve_ascii_emojis(self, text: str) -> str:
        """
        Replace ASCII emoticons with English labels (e.g. ':)' -> 'happy').
        """
        for pattern, label in self._ASCII_EMOJI_MAP:
            text = pattern.sub(f" {label} ", text)
        return text

    def normalise_for_classification(self, text: str) -> str:
        """
        Lowercase, expand contractions, resolve ASCII emojis, collapse elongation, strip non-alphanumeric.
        """
        text = text.lower()
        text = contractions.fix(text)                       # don't -> do not
        text = self.resolve_ascii_emojis(text)              # :) -> happy
        text = self._RE_ELONGATION.sub(r"\1\1", text)      # coooool -> cool
        text = self._RE_REPEATED_PUNCT.sub(r"\1", text)    # !!! -> !
        text = self._RE_NON_ALPHA.sub(" ", text)           # keep a-z, 0-9, spaces
        return re.sub(r"\s+", " ", text).strip()

    def drop_short(self, df: pd.DataFrame, text_col: str) -> pd.DataFrame:
        before = len(df)
        word_count = df[text_col].astype(str).str.split().str.len()
        out = df[word_count >= self.min_words]
        log.info(
            "Drop short (<%d words): %d -> %d rows (removed %d)",
            self.min_words,
            before,
            len(out),
            before - len(out),
        )
        return out

    def _emoji_to_english(self, text: str) -> str:
        """
        Translate emojis to short English labels
        """
        demojized = emoji.demojize(str(text), delimiters=(" ", " "))
        demojized = self._RE_MULTI_UNDERSCORE.sub("_", demojized)
        demojized = demojized.replace("_", " ")
        return demojized

    def process_emoji(self, df: pd.DataFrame, text_col: str) -> pd.DataFrame:
        out = df.copy()
        out["has_emoji"] = out[text_col].astype(str).apply(lambda t: bool(emoji.emoji_list(t)))
        out[text_col] = out[text_col].astype(str).apply(self._emoji_to_english).apply(self.normalise_text)
        return out

    @staticmethod
    def detect_language(text: str) -> str:
        try:
            return detect(str(text))
        except LangDetectException:
            return "unknown"

    def extract_features(self, df: pd.DataFrame, is_posts: bool, text_col_for_len_count: str = "cleaned_text") -> pd.DataFrame:
        out = df.copy()
        out["created_utc"] = pd.to_datetime(out["created_utc"], utc=True, errors="coerce")
        out["year"] = out["created_utc"].dt.year
        out["month"] = out["created_utc"].dt.month
        out["day_of_week"] = out["created_utc"].dt.day_name()
        out["hour"] = out["created_utc"].dt.hour
        out["score_bucket"] = pd.cut(out["score"].astype(float), bins=self.SCORE_BINS, labels=self.SCORE_LABELS)
        # Use the specified text_col_for_len_count for length and word count
        out["text_length"] = out[text_col_for_len_count].astype(str).str.len()
        out["word_count"] = out[text_col_for_len_count].astype(str).str.split().str.len()

        if is_posts:
            # Changed 'selftext' to 'content' as it was renamed earlier in the pipeline
            out["has_body"] = out["content"].notna() & (out["content"].astype(str).str.strip() != "")
        else:
            depth = out["depth"].fillna(0).astype(float)
            out["depth_bucket"] = pd.cut(
                depth, bins=[-1, 0, 2, 5, float("inf")], labels=["top_level", "shallow", "mid", "deep"]
            )
        return out

In [ ]:
reddit = RedditPreprocessor(min_words=3, lang_detect=False)

## Data Loading

In [ ]:
import os

# Define the path to your specific folder

folder_path = '/content/drive/MyDrive/CS5246Project'
data_folder_path = '/content/drive/MyDrive/CS5246Project/data/raw_data'

# Check if the folder exists and list its contents
if os.path.exists(folder_path):
    print(f"Contents of '{folder_path}':")
    for item in os.listdir(folder_path):
        print(item)
else:
    print(f"The folder '{folder_path}' does not exist. Please ensure the path is correct and the drive is mounted.")

Contents of '/content/drive/MyDrive/CS5246Project':
__pycache__
data
Labelling.ipynb
Stage_2_POS_and_NER_Tagging.ipynb
sentiment_plots
utilities
Stage_0_Introduction.ipynb
Stage_4_Singlish_to_English_Conversion.ipynb
Stage_3_Singlish_Normalisation.ipynb
Stage_5_Common_Normalisation.ipynb
Stage_6_Vector_Space_Model_and_Inverted_Index.ipynb
Stage_9_Document_Search.ipynb
Stage_8_Clustering_and_Visualization.ipynb
Stage_1_Data_Collection_and_Data_Cleaning.ipynb


In [ ]:
# Initialize an empty list to store the individual DataFrames
singapore_dfs = []

# Iterate through the files in the specified directory
for item in os.listdir(data_folder_path):
    file_path = os.path.join(data_folder_path, item)
    print(f"Attempting to read: {item}")
    df = pd.read_csv(file_path, encoding='utf-8',  on_bad_lines="skip",)
    try:
      singapore_dfs.append(df)
    except Exception as e:
      print(f"Error reading {item}: {e}")
      continue

    print(f"Successfully loaded '{item}' with UTF-8 encoding.")

print(f"\nTotal {len(singapore_dfs)} relevant DataFrame(s) loaded.")
if singapore_dfs:
    print("First few rows of the first loaded DataFrame:")
    display(singapore_dfs[0].head())

Attempting to read: singapore_posts_2026_01_january.csv
Successfully loaded 'singapore_posts_2026_01_january.csv' with UTF-8 encoding.
Attempting to read: singapore_comments_2026_01_january.csv
Successfully loaded 'singapore_comments_2026_01_january.csv' with UTF-8 encoding.
Attempting to read: singapore_posts_2025_01_january.csv
Successfully loaded 'singapore_posts_2025_01_january.csv' with UTF-8 encoding.
Attempting to read: singapore_comments_2025_01_january.csv


/tmp/ipykernel_7748/4149556292.py:8: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='utf-8',  on_bad_lines="skip",)


Successfully loaded 'singapore_comments_2025_01_january.csv' with UTF-8 encoding.
Attempting to read: singapore_posts_2025_02_february.csv
Successfully loaded 'singapore_posts_2025_02_february.csv' with UTF-8 encoding.
Attempting to read: singapore_comments_2025_02_february.csv
Successfully loaded 'singapore_comments_2025_02_february.csv' with UTF-8 encoding.
Attempting to read: singapore_comments_2025_03_march.csv


/tmp/ipykernel_7748/4149556292.py:8: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='utf-8',  on_bad_lines="skip",)


Successfully loaded 'singapore_comments_2025_03_march.csv' with UTF-8 encoding.
Attempting to read: singapore_posts_2025_03_march.csv
Successfully loaded 'singapore_posts_2025_03_march.csv' with UTF-8 encoding.
Attempting to read: singapore_comments_2025_04_april.csv
Successfully loaded 'singapore_comments_2025_04_april.csv' with UTF-8 encoding.
Attempting to read: singapore_posts_2025_04_april.csv
Successfully loaded 'singapore_posts_2025_04_april.csv' with UTF-8 encoding.
Attempting to read: singapore_comments_2025_05_may.csv


...

/tmp/ipykernel_7748/4149556292.py:8: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='utf-8',  on_bad_lines="skip",)


Successfully loaded 'singapore_comments_2025_09_september.csv' with UTF-8 encoding.
Attempting to read: singapore_posts_2025_09_september.csv
Successfully loaded 'singapore_posts_2025_09_september.csv' with UTF-8 encoding.
Attempting to read: singapore_posts_2025_10_october.csv
Successfully loaded 'singapore_posts_2025_10_october.csv' with UTF-8 encoding.
Attempting to read: singapore_comments_2025_10_october.csv
Successfully loaded 'singapore_comments_2025_10_october.csv' with UTF-8 encoding.
Attempting to read: singapore_posts_2025_11_november.csv
Successfully loaded 'singapore_posts_2025_11_november.csv' with UTF-8 encoding.
Attempting to read: singapore_comments_2025_11_november.csv
Successfully loaded 'singapore_comments_2025_11_november.csv' with UTF-8 encoding.
Attempting to read: singapore_posts_2025_12_december.csv
Successfully loaded 'singapore_posts_2025_12_december.csv' with UTF-8 encoding.
Attempting to read: singapore_comments_2025_12_december.csv


/tmp/ipykernel_7748/4149556292.py:8: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='utf-8',  on_bad_lines="skip",)


Successfully loaded 'singapore_comments_2025_12_december.csv' with UTF-8 encoding.
Attempting to read: singapore_posts_2026_02_february.csv
Successfully loaded 'singapore_posts_2026_02_february.csv' with UTF-8 encoding.
Attempting to read: singapore_comments_2026_02_february.csv
Successfully loaded 'singapore_comments_2026_02_february.csv' with UTF-8 encoding.

Total 28 relevant DataFrame(s) loaded.
First few rows of the first loaded DataFrame:


,id,title,selftext,author,score,upvote_ratio,num_comments,created_utc,edited,distinguished,...,author_flair_css_class,gilded,total_awards_received,treatment_tags,is_video,media,secure_media,thumbnail,post_hint,preview
0,1qsfnz9,r/singapore random discussion and small questi...,*🌻☀️Good morning all have a great day and stay...,AutoModerator,5,0.86,306,2026-01-31 22:00:52,NaN,NaN,...,NaN,0,0,[],False,NaN,NaN,self,NaN,NaN
1,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",NaN,Severe_County_5041,354,0.95,19,2026-01-31 16:21:49,NaN,NaN,...,NaN,0,0,[],False,NaN,NaN,default,NaN,NaN
2,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,NaN,btcprox,153,0.99,15,2026-01-31 15:37:29,NaN,NaN,...,NaN,0,0,[],False,NaN,NaN,https://external-preview.redd.it/8nVaG8OzdTP_o...,link,{'images': [{'source': {'url': 'https://extern...
3,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,_IsNull,87,0.89,24,2026-01-31 14:07:22,NaN,NaN,...,rainbow,0,0,[],False,NaN,NaN,https://external-preview.redd.it/YgofgeKcDsTut...,link,{'images': [{'source': {'url': 'https://extern...
4,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,Fit_Quit7002,182,0.96,17,2026-01-31 13:02:27,NaN,NaN,...,NaN,0,0,[],False,NaN,NaN,self,NaN,NaN


## Social Media Text Cleaning

Tasks:
1. Removing URLs
1. Stripping user mentions
1. Reducing character elongation
1. Normalizing punctuation
1. De-deduplicate
1. Remove bots and mods
1. Remove deleted posts
1. Remove 3 word posts
1. Expand contractions
1. Convert emoji to text

In [ ]:
#  Resolve ASCII emojis and expand contractions BEFORE clean_text runs
_PREPROCESS_COLS = ['title', 'selftext', 'body']
for df in singapore_dfs:
    for col in _PREPROCESS_COLS:
        if col in df.columns:
            # Only apply emoji-to-English conversion at this stage to raw columns.
            # Further normalization (contractions, ASCII emojis, lowercasing, etc.)
            # will be handled by reddit.normalise_for_classification later.
            df[col] = df[col].apply(
                lambda x: reddit._emoji_to_english(str(x)) if pd.notna(x) else x
            )

# Split the list of DataFrames into posts and comments based on column presence
post_dfs    = [df for df in singapore_dfs if 'title' in df.columns]
comment_dfs = [df for df in singapore_dfs if 'body'  in df.columns and 'title' not in df.columns]

# Concatenate the lists of DataFrames into single DataFrames for posts and comments
df_posts    = pd.concat(post_dfs,    ignore_index=True) if post_dfs    else pd.DataFrame()
df_comments = pd.concat(comment_dfs, ignore_index=True) if comment_dfs else pd.DataFrame()

# Define columns to keep for posts and comments explicitly
POST_COLS_TO_KEEP = [
    "id", "title", "content", # Keep original raw columns, 'selftext' renamed to 'content'
    "cleaned_title", "cleaned_content", # New cleaned columns
    "score", "upvote_ratio", "num_comments", "created_utc",
    "subreddit_id", "has_emoji", "year", "month", "day_of_week",
    "hour", "score_bucket", "text_length", "word_count"
]

COMMENT_COLS_TO_KEEP = [
    "id", "parent_id", "post_id", "text", "score", "created_utc", # 'body' renamed to 'content'
    "subreddit_id", "depth", "controversiality", "cleaned_text", "has_emoji",
    "year", "month", "day_of_week", "hour", "score_bucket", "text_length",
    "word_count", "depth_bucket"
]

# Process Posts DataFrame
if not df_posts.empty:
    print(f"Initial df_posts size: {len(df_posts)} rows")
    # Harmonize schema to ensure all expected columns are present
    df_posts = reddit.harmonise_schema(df_posts, reddit.POSTS_CANONICAL_COLS)
    print(f"df_posts after harmonise_schema: {len(df_posts)} rows")

    # Drop rows where 'id' is NaN
    before_drop_id = len(df_posts)
    df_posts.dropna(subset=['id'], inplace=True)
    print(f"df_posts after dropping NaN IDs: {len(df_posts)} (removed {before_drop_id - len(df_posts)}) rows")

    # Remove duplicate posts based on 'id'
    df_posts = reddit.dedup(df_posts, 'id')
    print(f"df_posts after dedup: {len(df_posts)} rows")
    # Drop posts that have been marked as 'removed' or 'deleted'
    df_posts = reddit.drop_removed(df_posts, 'title')
    print(f"df_posts after drop_removed (title): {len(df_posts)} rows")
    # Remove posts from known bots or moderators
    df_posts = reddit.drop_bots_and_mods(df_posts)
    print(f"df_posts after drop_bots_and_mods: {len(df_posts)} rows")

    # Rename 'selftext' to 'content' earlier in the pipeline
    df_posts = df_posts.rename(columns={"selftext": "content"})

    # Drop rows where both 'title' and 'content' are NaN (representing empty posts)
    before_drop_empty_post = len(df_posts)
    df_posts.dropna(subset=['title', 'content'], how='all', inplace=True)
    print(f"df_posts after dropping empty posts: {len(df_posts)} (removed {before_drop_empty_post - len(df_posts)}) rows")

    # Apply normalization for classification to 'title' and 'content' separately
    df_posts['cleaned_title'] = df_posts['title'].fillna('').str.strip().apply(reddit.normalise_for_classification)
    df_posts['cleaned_content'] = df_posts['content'].fillna('').str.strip().apply(reddit.normalise_for_classification)

    # Create a temporary combined cleaned text for post-level operations (like emoji detection, short-post filtering, and feature extraction)
    df_posts['temp_cleaned_text_for_post_ops'] = (
        df_posts['cleaned_title'].fillna('') + ' ' + df_posts['cleaned_content'].fillna('')
    ).str.strip()

    # Process emojis (adds 'has_emoji' and applies _emoji_to_english and normalise_text to the temporary combined text)
    df_posts = reddit.process_emoji(df_posts, 'temp_cleaned_text_for_post_ops')
    print(f"df_posts after process_emoji: {len(df_posts)} rows")
    # Remove posts with fewer words than the defined minimum (based on combined temporary text)
    df_posts = reddit.drop_short(df_posts, 'temp_cleaned_text_for_post_ops')
    print(f"df_posts after drop_short: {len(df_posts)} rows")
    # Extract additional features like time, score buckets, text length, and word count (uses temporary combined text)
    df_posts = reddit.extract_features(df_posts, is_posts=True, text_col_for_len_count='temp_cleaned_text_for_post_ops')
    print(f"df_posts after extract_features: {len(df_posts)} rows")

    # Drop the temporary combined text column as it's not part of the final output requirements
    df_posts.drop(columns=['temp_cleaned_text_for_post_ops'], inplace=True, errors='ignore')

    # Ensure 'title' and 'content' are empty strings if NaN, before final selection
    if 'title' in df_posts.columns:
        df_posts['title'] = df_posts['title'].fillna('')

    if 'cleaned_title' in df_posts.columns:
        df_posts['cleaned_title'] = df_posts['cleaned_title'].fillna('')


    if 'content' in df_posts.columns:
        df_posts['content'] = df_posts['content'].fillna('')

    if 'cleaned_content' in df_posts.columns:
        df_posts['cleaned_content'] = df_posts['cleaned_content'].fillna('')


    # Fill NaNs for specific columns before final selection
    numeric_cols_posts = ["score", "upvote_ratio", "num_comments", "text_length", "word_count", "year", "month", "hour"]
    categorical_cols_posts = ["day_of_week", "score_bucket", "subreddit_id"]
    boolean_cols_posts = ["has_emoji"]

    for col in numeric_cols_posts:
        if col in df_posts.columns:
            df_posts[col] = df_posts[col].fillna(0)
    for col in categorical_cols_posts:
        if col in df_posts.columns:
            
            if isinstance(df_posts[col].dtype, pd.CategoricalDtype):
                df_posts[col] = df_posts[col].astype(object)
            df_posts[col] = df_posts[col].fillna('unknown')
    for col in boolean_cols_posts:
        if col in df_posts.columns:
            df_posts[col] = df_posts[col].fillna(False)

    # Select only the desired columns
    df_posts = df_posts[POST_COLS_TO_KEEP]
    df_posts['fulltext'] = df_posts['title'] + ' ' + df_posts['content']
    print(f"Posts: {len(df_posts)} rows")

# Process Comments DataFrame
if not df_comments.empty:
    print(f"\nInitial df_comments size: {len(df_comments)} rows")
    # Harmonize schema to ensure all expected columns are present
    df_comments = reddit.harmonise_schema(df_comments, reddit.COMMENTS_CANONICAL_COLS)
    print(f"df_comments after harmonise_schema: {len(df_comments)} rows")

    # Drop rows where 'id' is NaN
    before_drop_id = len(df_comments)
    df_comments.dropna(subset=['id'], inplace=True)
    print(f"df_comments after dropping NaN IDs: {len(df_comments)} (removed {before_drop_id - len(df_comments)}) rows")

    # Remove duplicate comments based on 'id'
    df_comments = reddit.dedup(df_comments, 'id')
    print(f"df_comments after dedup: {len(df_comments)} rows")
    # Drop comments that have been marked as 'removed' or 'deleted'
    df_comments = reddit.drop_removed(df_comments, 'body')
    print(f"df_comments after drop_removed (body): {len(df_comments)} rows")
    # Remove comments from known bots or moderators
    df_comments = reddit.drop_bots_and_mods(df_comments)
    print(f"df_comments after drop_bots_and_mods: {len(df_comments)} rows")

    # Rename 'body' to 'text' earlier in the pipeline
    df_comments = df_comments.rename(columns={"body": "text"})

    # Drop rows where 'text' is NaN (representing empty comments)
    before_drop_empty_comment = len(df_comments)
    df_comments.dropna(subset=['text'], inplace=True)
    print(f"df_comments after dropping empty comments: {len(df_comments)} (removed {before_drop_empty_comment - len(df_comments)}) rows")

    # Create a clean parent_id (used for some merges or identification)
    df_comments['parent_id'] = df_comments['parent_id'].str.replace('t3_', '')

    # Apply normalization for classification to the pre-cleaned 'text' column
    df_comments['cleaned_text'] = (
        df_comments['text'].fillna('').str.strip().apply(reddit.normalise_for_classification)
    )
    # Process emojis again to ensure consistency
    df_comments = reddit.process_emoji(df_comments, 'cleaned_text')
    print(f"df_comments after process_emoji: {len(df_comments)} rows")
    # Remove comments with fewer words than the defined minimum
    df_comments = reddit.drop_short(df_comments, 'cleaned_text')
    print(f"df_comments after drop_short: {len(df_comments)} rows")
    # Extract additional features like time, score buckets, and text length/word count
    df_comments = reddit.extract_features(df_comments, is_posts=False, text_col_for_len_count='cleaned_text')
    print(f"df_comments after extract_features: {len(df_comments)} rows")

    # Ensure 'content' and 'cleaned_text' are empty strings if NaN, before final selection
    if 'content' in df_comments.columns:
        df_comments['content'] = df_comments['content'].fillna('')

    if 'cleaned_text' in df_comments.columns:
        df_comments['cleaned_text'] = df_comments['cleaned_text'].fillna('')

    # Fill NaNs for specific columns before final selection
    numeric_cols_comments = ["score", "depth", "controversiality", "text_length", "word_count", "year", "month", "hour"]
    categorical_cols_comments = ["day_of_week", "score_bucket", "depth_bucket", "subreddit_id"]
    boolean_cols_comments = ["has_emoji"]

    for col in numeric_cols_comments:
        if col in df_comments.columns:
            df_comments[col] = df_comments[col].fillna(0)
    for col in categorical_cols_comments:
        if col in df_comments.columns:
            if isinstance(df_comments[col].dtype, pd.CategoricalDtype):
                df_comments[col] = df_comments[col].astype(object)
            df_comments[col] = df_comments[col].fillna('unknown')
    for col in boolean_cols_comments:
        if col in df_comments.columns:
            df_comments[col] = df_comments[col].fillna(False)

    # Select only the desired columns
    df_comments = df_comments[COMMENT_COLS_TO_KEEP]

    print(f"Comments: {len(df_comments)} rows")

Initial df_posts size: 32926 rows
df_posts after harmonise_schema: 32926 rows
df_posts after dropping NaN IDs: 32926 (removed 0) rows
df_posts after dedup: 32926 rows
df_posts after drop_removed (title): 32926 rows
df_posts after drop_bots_and_mods: 32497 rows
df_posts after dropping empty posts: 32497 (removed 0) rows
df_posts after process_emoji: 32497 rows
df_posts after drop_short: 31957 rows
df_posts after extract_features: 31957 rows
Posts: 31957 rows

Initial df_comments size: 825042 rows
df_comments after harmonise_schema: 825042 rows
df_comments after dropping NaN IDs: 825042 (removed 0) rows
df_comments after dedup: 825042 rows
df_comments after drop_removed (body): 801304 rows
df_comments after drop_bots_and_mods: 795827 rows
df_comments after dropping empty comments: 795827 (removed 0) rows
df_comments after process_emoji: 795827 rows
df_comments after drop_short: 764462 rows
df_comments after extract_features: 764462 rows
Comments: 764462 rows


In [ ]:
comments = df_comments.copy()
print("-------------------------------------------------")
print("First few rows of comments['cleaned_text'] column")
print("-------------------------------------------------")
display(comments[['text', 'cleaned_text']].head())
print()

posts = df_posts.copy()
print("-----------------------------------------------------------------------------")
print("First few rows of posts['cleaned_title'] and posts['cleaned_content'] columns")
print("-----------------------------------------------------------------------------")
display(posts[['title', 'content', 'cleaned_title', 'cleaned_content']].head())

-------------------------------------------------
First few rows of comments['cleaned_text'] column
-------------------------------------------------


,text,cleaned_text
0,she said yes! :D,she said yes grinning
1,Got an in principle approval for Singapore cit...,got an in principle approval for singapore cit...
2,I am reading the Jeffrey Epstein files. I need...,i am reading the jeffrey epstein files i need ...
3,Wonderful sleep last night.. haven't slept so ...,wonderful sleep last night have not slept so m...
4,![gif](giphy|1luXLMeNxsaNFMUuOe),gif giphy 1luxlmenxsanfmuuoe happy



-----------------------------------------------------------------------------
First few rows of posts['cleaned_title'] and posts['cleaned_content'] columns
-----------------------------------------------------------------------------


,title,content,cleaned_title,cleaned_content
1,"A Hot Hideout mala chain co-founder, 26, diagn...",,a hot hideout mala chain co founder 26 diagnos...,
2,Fantasy meets whimsy at medieval-themed renais...,,fantasy meets whimsy at medieval themed renais...,
3,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,pair of adjoining ground floor hdb shops in pa...,the property has a remaining tenure of 63 year...
4,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,video call scam targeting elderly,my 90 year old mum just got a video call from ...
5,Woodlands Checkpoint officer dragged by motori...,,woodlands checkpoint officer dragged by motori...,


### Load Files

In [ ]:
df_posts = pd.read_csv("/content/drive/MyDrive/CS5246Project/data/PostVault.csv", keep_default_na=False)
df_comments = pd.read_csv("/content/drive/MyDrive/CS5246Project/data/CommentVault.csv", keep_default_na=False)

### Save Files

In [ ]:
# ---------
# PostVault
# ---------
df_posts = df_posts.reindex(posts.index)
for col in posts.columns:
    df_posts.loc[:, col] = posts[col]

display(df_posts.head())
print("-----------")
print("Posts Shape")
print("-----------")
print(df_posts.shape)
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/CS5246Project/data/", "PostVault")

# ------------
# CommentVault
# ------------
df_comments = df_comments.reindex(comments.index)
for col in comments.columns:
    df_comments.loc[:, col] = comments[col]

display(df_comments.head())
print("--------------")
print("Comments Shape")
print("--------------")
print(df_comments.shape)
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/CS5246Project/data/", "CommentVault")

,id,title,content,cleaned_title,cleaned_content,score,upvote_ratio,num_comments,created_utc,subreddit_id,...,english_converted_content,expanded_title,expanded_content,demojized_title,demojized_content,spellchecked_title,spellchecked_content,lemmatized_title,lemmatized_content,tfidf_cluster
1,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",,a hot hideout mala chain co founder 26 diagnos...,,354.0,0.95,19.0,2026-01-31 16:21:49+00:00,t5_2qh8c,...,,fantasy meets whimsy at medieval themed renais...,,fantasy meets whimsy at medieval themed renais...,,fantasy meets whimsy at medieval themed renais...,,fantasy meet whimsy medieval themed renaissanc...,,6.0
2,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,,fantasy meets whimsy at medieval themed renais...,,153.0,0.99,15.0,2026-01-31 15:37:29+00:00,t5_2qh8c,...,the property has a remaining tenure of 63 year...,pair of adjoining ground floor hdb shops in pa...,the property has a remaining tenure of 63 year...,pair of adjoining ground floor hdb shops in pa...,the property has a remaining tenure of 63 year...,pair of adjoining ground floor hub shops in pa...,the property has a remaining tenure of 63 year...,pair adjoining ground floor hdb shop pasir ri ...,property remaining tenure 63 year available sa...,6.0
3,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,pair of adjoining ground floor hdb shops in pa...,the property has a remaining tenure of 63 year...,87.0,0.89,24.0,2026-01-31 14:07:22+00:00,t5_2qh8c,...,my 90 year old mum just got a video call from ...,video call scam targeting elderly,my 90 year old mum just got a video call from ...,video call scam targeting elderly,my 90 year old mum just got a video call from ...,video call scam targeting elderly,my 90 year old mum just got a video call from ...,video call scam targeting elderly,90 year old mum got video call someone full po...,6.0
4,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,video call scam targeting elderly,my 90 year old mum just got a video call from ...,182.0,0.96,17.0,2026-01-31 13:02:27+00:00,t5_2qh8c,...,,woodlands checkpoint officer dragged by motori...,,woodlands checkpoint officer dragged by motori...,,woodlands checkpoint officer dragged by motori...,,woodland checkpoint officer dragged motorist f...,,1.0
5,1qrzbgd,Woodlands Checkpoint officer dragged by motori...,,woodlands checkpoint officer dragged by motori...,,104.0,0.92,20.0,2026-01-31 10:51:19+00:00,t5_2qh8c,...,,bangkok post singapore a transit point for uk ...,,bangkok post singapore a transit point for uk ...,,bangkok post singapore a transit point for up ...,,bangkok post singapore transit point uk bound ...,,1.0


-----------
Posts Shape
-----------
(31957, 32)
Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data/' with name 'PostVault'...
----------------------------------------------------------------------------------------------------
PostVault saved to /content/drive/MyDrive/CS5246Project/data/PostVault.csv
----------------------------------------------------------------------------------------------------


,id,parent_id,post_id,text,score,created_utc,subreddit_id,depth,controversiality,cleaned_text,has_emoji,year,month,day_of_week,hour,score_bucket,text_length,word_count,depth_bucket
0,o2xddjd,1qsfnz9,1qsfnz9,she said yes! :D,26.0,2026-02-01 06:04:53+00:00,t5_2qh8c,0.0,0.0,she said yes grinning,False,2026.0,2.0,Sunday,6.0,high,21.0,4.0,top_level
1,o2wlc4e,1qsfnz9,1qsfnz9,Got an in principle approval for Singapore cit...,16.0,2026-02-01 02:53:11+00:00,t5_2qh8c,0.0,0.0,got an in principle approval for singapore cit...,False,2026.0,2.0,Sunday,2.0,medium,129.0,24.0,top_level
2,o2w9wny,1qsfnz9,1qsfnz9,I am reading the Jeffrey Epstein files. I need...,8.0,2026-02-01 01:44:30+00:00,t5_2qh8c,0.0,0.0,i am reading the jeffrey epstein files i need ...,False,2026.0,2.0,Sunday,1.0,medium,133.0,26.0,top_level
3,o2vkfvf,1qsfnz9,1qsfnz9,Wonderful sleep last night.. haven't slept so ...,7.0,2026-01-31 23:18:42+00:00,t5_2qh8c,0.0,0.0,wonderful sleep last night have not slept so m...,False,2026.0,1.0,Saturday,23.0,medium,134.0,26.0,top_level
4,o2xk0qh,1qsfnz9,1qsfnz9,![gif](giphy|1luXLMeNxsaNFMUuOe),6.0,2026-02-01 07:00:47+00:00,t5_2qh8c,0.0,0.0,gif giphy 1luxlmenxsanfmuuoe happy,False,2026.0,2.0,Sunday,7.0,medium,34.0,4.0,top_level


--------------
Comments Shape
--------------
(764462, 19)
Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data/' with name 'CommentVault'...
----------------------------------------------------------------------------------------------------
CommentVault saved to /content/drive/MyDrive/CS5246Project/data/CommentVault.csv
----------------------------------------------------------------------------------------------------
